## Environment Setup and Library Import

This section installs and imports all the required libraries used throughout the notebook.  
The implementation relies on PyTorch for deep learning operations, OpenCV and PIL for image processing, and additional utility libraries for data handling and evaluation.

The `ood_metrics` package is used to compute common Out-of-Distribution (OOD) detection metrics, while `scikit-learn` provides functions for performance evaluation such as the Average Precision score.

In [9]:
import os
import sys
import glob
import torch
import random
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score
import warnings


## Repository Setup

This section clones the project repository and configures the working environment for the anomaly detection experiments.

The repository contains the implementation of the ERFNet architecture together with utilities for Out-of-Distribution (OOD) evaluation.  
The repository path is added to the Python environment in order to directly import the required custom modules.

In [ ]:
# Setup path containing the evaluation scripts and custom models
repo_path = "./" # Change the repo path
if os.path.exists(repo_path):
    sys.path.append(repo_path)
    from erfnet import ERFNet
    from ood_metrics import fpr_at_95_tpr
else:
    print("Repository does not exist")

## Reproducibility and Data Preprocessing

This section defines the random seed configuration and the preprocessing pipeline used for both input images and target labels.

To ensure reproducibility of the experiments, the random seed is fixed for Python, NumPy, and PyTorch.  
In addition, specific CUDA backend settings are configured to reduce non-deterministic behavior during execution.

The preprocessing pipeline resizes all images to the resolution required by the network and converts them into tensors suitable for PyTorch processing.  
Different interpolation methods are used for input images and segmentation masks in order to preserve the semantic integrity of the labels.

In [6]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)

## Anomaly Score Computation and Ground Truth Preprocessing

This section defines the functions used to compute anomaly scores from the network output logits and to preprocess the ground truth annotations for different OOD datasets.

Different scoring strategies are considered for anomaly detection:

- **MSP (Maximum Softmax Probability)**: anomalies are identified through low confidence predictions;
- **MaxLogit**: uses the maximum raw logit value instead of probabilities;
- **Entropy**: measures the uncertainty of the predictive distribution.

In addition, since each OOD benchmark adopts different label conventions, a dedicated preprocessing function is used to remap the annotations into a unified binary format:
- `0` → in-distribution pixels,
- `1` → out-of-distribution pixels,
- `255` → ignored regions.

In [7]:
def compute_anomaly_score(logits, method="MSP", num_classes=20):
    """Compute the anomaly score from the network output logits.
    """
    # Remove batch size (1, C, H, W) -> (C, H, W)
    if logits.dim() == 4 and logits.shape[0] == 1:
        logits = logits.squeeze(0)

    # Softmax probabilities are required for MSP and Entropy (dim=0 perché abbiamo rimosso il batch)
    if method in ["MSP", "Entropy"]:
        probs = torch.softmax(logits, dim=0)

    if method == "MSP":
        # MSP: 1 - max probability
        max_probs, _ = torch.max(probs, dim=0)
        return 1.0 - max_probs.cpu().numpy()

    elif method == "MaxLogit":
        # MaxLogit: - max logit (teh lower is logit, the higher is the anomaly)
        max_logits, _ = torch.max(logits, dim=0)
        return - max_logits.cpu().numpy()

    elif method == "Entropy":
        # Normalized Entropy: H / log(N) with range [0, 1]
        # 1e-10 for numerical stability
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=0)

        # Normalizzation
        log_num_classes = torch.log(torch.tensor(num_classes, dtype=torch.float32))
        normalized_entropy = entropy / log_num_classes

        return normalized_entropy.cpu().numpy()

    return None

def preprocess_gt(path_gt, mask_img):
    """ Remap dataset-specific labels into a unified binary OOD fromat.
    Output convention:
     0 --> in-distribution
     1 --> out-of-distribution
     255--> ignored pixels
    """
    ood_gts = np.array(mask_img)

    # RoadAnomaly 21 already follows the desired fromat
    if "RoadAnomaly21" in path_gt:
        pass
    # RoadAnomaly label remapping
    if "RoadAnomaly" in path_gt:
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)
    # LostAndFound label remapping
    elif "LostAndFound" in path_gt:
        ood_gts = np.where((ood_gts == 0), 255, ood_gts)
        ood_gts = np.where((ood_gts == 1), 0, ood_gts)
        ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)

    return ood_gts


## Model Initialization and Pretrained Weight Loading

This section initializes the ERFNet semantic segmentation model and loads the pretrained weights used for inference.

The network is configured for multi-class semantic segmentation and wrapped with `DataParallel` in order to enable GPU-based parallel execution.  
Pretrained weights are then loaded from a checkpoint file. Since checkpoints may contain different naming conventions depending on the training configuration, an additional compatibility step is included to correctly map the parameters into the current model architecture.

Finally, the model is switched to evaluation mode to disable training-specific behaviors such as dropout and batch normalization updates.

In [ ]:
# Initialize the ERNet semantic segmentation architecture
model = ERFNet(NUM_CLASSES)
model = torch.nn.DataParallel(model).cuda()

# Path to the pretrained model weights
weights_path = "../trained_models/erfnet_pretrained.pth" # Change the weight path
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    checkpoint = torch.load(weights_path, map_location='cuda')


if os.path.exists(weights_path):
    checkpoint = torch.load(weights_path, map_location='cuda')
    own_state = model.state_dict()
    for name, param in checkpoint.items():
        if name in own_state: own_state[name].copy_(param)
        elif name.startswith("module."): own_state[name.split("module.")[-1]].copy_(param)
    print("Model and prertained weights loaded successfully.")
else:
    print("WARNING: Pretrained weight file not found!")

model.eval()


Model and prertained weights loaded successfully.


C:\Users\user\AppData\Local\Temp\ipykernel_14624\3850880171.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weights_path, map_location='cuda')


DataParallel(
  (module): ERFNet(
    (encoder): Encoder(
      (initial_block): DownsamplerBlock(
        (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (layers): ModuleList(
        (0): DownsamplerBlock(
          (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1-5): 5 x non_bottleneck_1d(
          (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
          (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
          (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True

## OOD Evaluation Pipeline

This section performs the complete evaluation pipeline for anomaly detection across multiple benchmark datasets.

For each dataset, the model processes all input images and computes pixel-wise anomaly scores using different scoring strategies:
- Maximum Softmax Probability (MSP),
- Maximum Logit,
- Entropy-based uncertainty.

The predicted anomaly maps are then compared against the corresponding ground truth annotations after dataset-specific preprocessing.

Finally, two standard OOD detection metrics are computed:
- **AUPRC (Area Under the Precision-Recall Curve)**, which evaluates the quality of anomaly ranking;
- **FPR95 (False Positive Rate at 95% True Positive Rate)**, commonly used to measure the trade-off between sensitivity and false alarms in OOD detection tasks.

The results obtained for each dataset and scoring method are printed and stored for later analysis.

In [18]:
# Definition of the evaluation datasets
BASE = "../Anomaly_Validation_Datasets/Validation_Dataset" # Change the Anomaly_Validation_Datasets path
datasets = [

    {
        "name": "RoadAnomaly21",
        "path": f"{BASE}/RoadAnomaly21/images/*.png"
    },

    {
        "name": "RoadObstacle21",
        "path": f"{BASE}/RoadObsticle21/images/*.webp"
    },

    {
        "name": "FS_Lost&Found",
        "path": f"{BASE}/FS_LostFound_full/images/*.png"
    },

    {
        "name": "FS_Static",
        "path": f"{BASE}/fs_static/images/*.jpg"
    },

    {
        "name": "RoadAnomaly",
        "path": f"{BASE}RoadAnomaly/images/*.jpg"
    },
]

# List of anomaly scoring methods
methods = ["MSP", "MaxLogit", "Entropy"]
# File used to store evaluation results
results_file_path = 'results.txt'

# Iterate over all evaluation datasets
for ds in datasets:
    print(f"\n" + "="*50)
    print(f" Dataset Analysis: {ds['name']}")
    print("="*50)

    # Evaluate all anomaly scoring methods
    for method in methods:
        scores_list = []
        gts_list = []

        # Retrive all dataset images
        input_files = glob.glob(os.path.expanduser(ds['path']))
        if not input_files: continue

        # Process each image independently
        for path in input_files:
            if os.path.isdir(path): continue

            img_raw = Image.open(path).convert('RGB')
            img_tensor = input_transform(img_raw).unsqueeze(0).cuda()

            # Model inference
            with torch.no_grad():
                logits = model(img_tensor)
                score = compute_anomaly_score(logits, method=method, num_classes=NUM_CLASSES)

            # Ground Truth loading
            pathGT = path.replace("images", "labels_masks").replace("webp", "png").replace("jpg", "png")
            # Verify that the ground truth exists
            if os.path.exists(pathGT):
                mask = target_transform(Image.open(pathGT))
                gt = preprocess_gt(pathGT, mask)
                if 1 in np.unique(gt):
                    scores_list.append(score)
                    gts_list.append(gt)

            torch.cuda.empty_cache()

        # -Metric computation
        if scores_list:
            all_scores = np.array(scores_list)
            all_gts = np.array(gts_list)

            # Binary masks for ID and OOD pixels
            ind_mask = (all_gts == 0)
            ood_mask = (all_gts == 1)

            # Construct binary labels
            y_true = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))
            # Construct anomaly scores
            y_scores = np.concatenate((all_scores[ind_mask], all_scores[ood_mask]))

            # Compute evaluation metrics
            auprc = average_precision_score(y_true, y_scores)
            fpr95 = fpr_at_95_tpr(y_scores, y_true)

            # Log a video
            result_text = f"Dataset: {ds['name']:15} | Method: {method:10} | AUPRC: {auprc*100.0:.2f}% | FPR95: {fpr95*100.0:.2f}%"
            print(result_text)

            # Save results to file
            with open(results_file_path, 'a') as f:
                f.write(result_text + "\n")


print(
    f"\nEvaluation completed. "
    f"Results saved to {results_file_path}"
)


 Dataset Analysis: RoadAnomaly21
Dataset: RoadAnomaly21   | Method: MSP        | AUPRC: 29.09% | FPR95: 62.62%
Dataset: RoadAnomaly21   | Method: MaxLogit   | AUPRC: 38.29% | FPR95: 59.42%
Dataset: RoadAnomaly21   | Method: Entropy    | AUPRC: 30.95% | FPR95: 62.74%

 Dataset Analysis: RoadObstacle21
Dataset: RoadObstacle21  | Method: MSP        | AUPRC: 2.71% | FPR95: 65.27%
Dataset: RoadObstacle21  | Method: MaxLogit   | AUPRC: 4.63% | FPR95: 48.47%
Dataset: RoadObstacle21  | Method: Entropy    | AUPRC: 3.04% | FPR95: 65.94%

 Dataset Analysis: FS_Lost&Found
Dataset: FS_Lost&Found   | Method: MSP        | AUPRC: 1.75% | FPR95: 50.61%
Dataset: FS_Lost&Found   | Method: MaxLogit   | AUPRC: 3.30% | FPR95: 45.51%
Dataset: FS_Lost&Found   | Method: Entropy    | AUPRC: 2.58% | FPR95: 50.19%

 Dataset Analysis: FS_Static
Dataset: FS_Static       | Method: MSP        | AUPRC: 7.43% | FPR95: 41.84%
Dataset: FS_Static       | Method: MaxLogit   | AUPRC: 9.44% | FPR95: 40.31%
Dataset: FS_Stati

## Final Considerations

The obtained results highlight the intrinsic difficulty of Out-of-Distribution (OOD) detection in real-world driving scenarios using standard semantic segmentation confidence-based methods.

Among the evaluated scoring strategies, **MaxLogit** consistently achieved the best performance across all datasets, providing higher AUPRC values and lower FPR95 scores compared to MSP and Entropy-based approaches.  
This suggests that raw logits preserve more discriminative information for anomaly detection than probability-based confidence measures derived from the softmax distribution.

The best overall performance was observed on the **RoadAnomaly21** dataset, where MaxLogit reached an AUPRC of approximately 38.3% with an FPR95 of about 59.3%.  
Although these values remain relatively modest, they indicate that the model is still capable of partially distinguishing anomalous regions from in-distribution content in structured road environments.

Conversely, performance significantly decreases on more challenging datasets such as **RoadObstacle21** and **FS Lost&Found**, where the AUPRC values remain below 5%.  
These results suggest that the pretrained ERFNet model struggles to reliably identify small, rare, or visually ambiguous obstacles when relying exclusively on confidence-based anomaly estimation.

The generally high FPR95 values across all benchmarks further indicate that the model produces a considerable number of false positives when operating at high recall levels.  
This behavior is common in OOD segmentation tasks, especially when the network has not been explicitly trained for anomaly detection.

Overall, the experiments show that simple post-processing strategies based on semantic segmentation confidence are insufficient for robust anomaly segmentation in complex driving environments.  
More advanced approaches — such as feature-space modeling, energy-based methods, synthetic anomaly exposure, or dedicated OOD training objectives — would likely be necessary to achieve reliable real-world performance.